# 硬负样本挖掘

硬负样本是指那些对模型来说特别难以与正样本区分开的负样本。它们通常接近决策边界，或者具有使它们与正样本高度相似的特征。因此，硬负样本挖掘在机器学习任务中被广泛使用，使模型能够专注于相似实例之间的微妙差异，从而获得更好的判别能力。

在文本检索系统中，硬负样本可能是那些与查询在特征上有一些相似性，但实际上并不能真正满足查询意图的文档。在检索过程中，这些文档可能会比真正的答案排名更高。因此，明确地让模型学习这些硬负样本是很有价值的。

## 1. 准备工作

首先，加载一个嵌入模型：

In [ ]:
from FlagEmbedding import FlagModel

model = FlagModel('BAAI/bge-base-en-v1.5')

然后，从数据集中加载查询和语料库：

In [ ]:
from datasets import load_dataset

corpus = load_dataset("BeIR/scifact", "corpus")["corpus"]
queries = load_dataset("BeIR/scifact", "queries")["queries"]

corpus_ids = corpus.select_columns(["_id"])["_id"]
corpus = corpus.select_columns(["text"])["text"]

我们创建一个字典，将FAISS索引使用的自动生成的ID（从0开始）映射到原始ID，以便后续使用。

In [ ]:
corpus_ids_map = {}
for i in range(len(corpus)):
    corpus_ids_map[i] = corpus_ids[i]

## 2. 索引建立

使用嵌入模型来编码查询和语料库：

In [ ]:
p_vecs = model.encode(corpus)

In [ ]:
p_vecs.shape

然后创建一个FAISS索引

In [ ]:
import torch, faiss
import numpy as np

# create a basic flat index with dimension match our embedding
index = faiss.IndexFlatIP(len(p_vecs[0]))
# make sure the embeddings are float32
p_vecs = np.asarray(p_vecs, dtype=np.float32)
# use gpu to accelerate index searching
if torch.cuda.is_available():
    co = faiss.GpuMultipleClonerOptions()
    co.shard = True
    co.useFloat16 = True
    index = faiss.index_cpu_to_all_gpus(index, co=co)
# add all the embeddings to the index
index.add(p_vecs)

## 3. 搜索

为了更好地演示，让我们使用单个查询：

In [ ]:
query = queries[0]
query

获取该查询的ID和内容，然后使用我们的嵌入模型获取其嵌入向量。

In [ ]:
q_id, q_text = query["_id"], query["text"]
# use the encode_queries() function to encode query
q_vec = model.encode_queries(queries=q_text)

使用索引搜索最接近的结果：

In [ ]:
_, ids = index.search(np.expand_dims(q_vec, axis=0), k=15)
# convert the auto ids back to ids in the original dataset
converted = [corpus_ids_map[id] for id in ids[0]]
converted

In [ ]:
qrels = load_dataset("BeIR/scifact-qrels")["train"]
pos_id = qrels[0]
pos_id

最后，我们使用"前K个结果偏移N位"的方法，获取排名5之后的前10个负样本。

In [ ]:
negatives = [id for id in converted[5:] if int(id) != pos_id["corpus-id"]]
negatives

现在我们已经为第一个查询选择了一组硬负样本！

还有其他方法可以完善选择硬负样本的过程。例如，我们GitHub仓库中的[实现](https://github.com/FlagOpen/FlagEmbedding/blob/master/scripts/hn_mine.py)获取了偏移10位后的前200个结果，即排名10-210的结果。然后从这200个候选中抽样15个。这样做的原因是直接选择前K个可能会在负样本集中引入一些假负样本，即某种程度上与查询相关但不完全是该查询答案的段落。这可能会影响模型的性能。